# 8.5 · XGBoost / LightGBM / CatBoost 对比 / The Boosting Big Three

> **课程定位 / Where this fits**
> 第 5 课，**Part 8 · 集成学习**。
> Lesson 5, **Part 8 · Ensemble Learning**.
>
> 8.4 讲了 GBDT 的通用原理。现实中没人手写 GBDT——大家用三大高度优化的库：**XGBoost、LightGBM、CatBoost**。它们都基于梯度提升(8.4)，但在**树怎么长、怎么加速、怎么处理类别特征**上各有绝活。5.9-5.11 分别介绍过，这一课**横向对比**：同一数据上比速度/精度，并给出"何时用哪个"的实战指南。这是表格数据建模的**最终选型课**。
> 8.4 covered GBDT's general principle. Nobody hand-writes GBDT in practice — everyone uses three highly-optimized libraries: **XGBoost, LightGBM, CatBoost**. All are gradient boosting (8.4), but each has tricks in **how trees grow, how they accelerate, and how they handle categorical features**. Introduced individually in 5.9-5.11, this lesson **compares them head-to-head**: speed/accuracy on one dataset, plus a practical "when to use which" guide. The final model-selection lesson for tabular data.
>
> 💼 **实战/面试视角**："三大 boosting 库区别 / leaf-wise vs level-wise / 类别特征怎么处理 / 怎么选" 高频。
> 💼 **Practical/interview angle:** "differences among the three / leaf-wise vs level-wise / categorical handling / how to choose" — common.

> 💡 **面试相关 / Interview-relevant**
> - "XGBoost/LightGBM/CatBoost 三者的核心区别"（出镜率 ★★★★★）
> - "leaf-wise vs level-wise 生长"（★★★★★）
> - "LightGBM 为什么快（直方图+GOSS+EFB）"（★★★★）
> - "CatBoost 怎么处理类别特征（有序TS防泄漏）"（★★★★）
> - "实战怎么在三者间选"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解三者的核心区别（生长策略、加速、类别处理）。
   Understand their core differences (growth, acceleration, categorical handling).
2. 在同一数据上横向比较速度与精度。
   Compare speed and accuracy on the same data.
3. 看 CatBoost 原生处理类别特征的威力。
   See CatBoost's native categorical handling in action.
4. 掌握"何时用哪个"的实战选型。
   Master the practical "which to use when".

## 目录 / TOC
1. [先建直觉：三者的绝活 ⭐](#1)
2. [💰 数据（含类别特征）](#2)
3. [横向对比：速度 + 精度 ⭐](#3)
4. [CatBoost 原生类别特征 ⭐](#4)
5. [选型指南 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：三者的绝活 ⭐ / Intuition: Each One's Trick

三者都在做 8.4 的梯度提升，区别在工程与算法细节（面试核心对照表）：
All three do the gradient boosting of 8.4; they differ in engineering and algorithmic details (the core comparison table):

| | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| 树生长 growth | level-wise（按层，平衡）| **leaf-wise**（按叶最大增益，更深更准但易过拟合）| **对称树**（每层同一分裂，强正则+预测快）|
| 加速核心 speed | 二阶泰勒 + 直方图 | **直方图 + GOSS + EFB**（通常最快）| 对称树 + GPU |
| 类别特征 categorical | 需编码（新版有原生）| 原生（直方图分割）| **原生（有序目标统计, 防泄漏）** |
| 招牌优势 | 通用霸主、生态成熟 | 大数据/高维**最快** | **类别特征多、默认参数就强** |

一句话记忆：**XGBoost 稳、LightGBM 快、CatBoost 擅类别**。
One-liner: **XGBoost = solid, LightGBM = fast, CatBoost = great with categoricals.**


<a id="2"></a>
## 2. 数据（含类别特征）/ Data with a Categorical Feature

合成 **Adult Income**（同 5.8/5.11 风格），并加入一个**有真实信号的类别特征 occupation**（不同职业有不同高收入倾向）。这样能公平对比三者，尤其展示 CatBoost 的类别处理。
Synthetic **Adult Income** (5.8/5.11 style) plus a **signal-bearing categorical feature `occupation`** (different jobs lean differently toward high income). This fairly compares all three and showcases CatBoost's categorical handling.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid")

def make_income_cat(n=20000, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, n); edu_years = rng.integers(6, 21, n)
    hours = rng.normal(40, 10, n).clip(10, 80)
    capital_gain = (rng.random(n) < 0.15) * rng.exponential(5000, n)
    occ = rng.choice(["tech","mgmt","sales","admin","service"], n, p=[.2,.15,.25,.2,.2])
    occ_effect = {"tech":1.2,"mgmt":1.5,"sales":0.2,"admin":-0.3,"service":-0.8}   # 职业带真实信号
    bonus = np.array([occ_effect[o] for o in occ])
    logit = (-9 + 0.04*age - 0.0004*(age-45)**2 + 0.25*edu_years + 0.02*hours
             + 0.0002*np.sqrt(capital_gain)*edu_years*0.3 + bonus + rng.normal(0, 0.5, n))
    y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
    X = pd.DataFrame({"age": age, "edu_years": edu_years, "hours": hours,
                      "capital_gain": capital_gain.round(0), "occupation": occ})
    return X, y

X, y = make_income_cat()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(f"合成 Adult Income(含类别): {X.shape}, 高收入率 {y.mean():.0%}")
print("各职业高收入率:"); print(pd.crosstab(X.occupation, y, normalize='index')[1].round(2).to_string())


<a id="3"></a>
## 3. 横向对比：速度 + 精度 ⭐ / Head-to-head: Speed + Accuracy

在同一数据上跑三者（类别特征先 one-hot 让三者公平起跑），测训练时间和 test AUC。注意：**结果会受数据规模、特征数、参数影响**——别把某一次的排名当绝对真理，重点理解"为什么会有差异"。
Run all three on the same data (one-hot the categorical for a fair start), measuring training time and test AUC. Note: **results depend on data size, feature count, and parameters** — don't take one run's ranking as gospel; focus on understanding "why differences arise".


In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 公平起跑: 三者都用 one-hot 后的特征 / one-hot for a fair comparison
ct = ColumnTransformer([("oh", OneHotEncoder(handle_unknown="ignore"), ["occupation"])], remainder="passthrough")
Xtr_oh = ct.fit_transform(X_tr); Xte_oh = ct.transform(X_te)

models = {
    "XGBoost":  xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, eval_metric="auc", random_state=0),
    "LightGBM": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=0, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=300, learning_rate=0.05, depth=5, verbose=0, random_seed=0),
}
print(f"{'库 library':<12}{'训练秒 train s':>14}{'test AUC':>10}")
for name, m in models.items():
    t = time.perf_counter(); m.fit(Xtr_oh, y_tr); dt = time.perf_counter()-t
    auc = roc_auc_score(y_te, m.predict_proba(Xte_oh)[:,1])
    print(f"{name:<12}{dt:>14.2f}{auc:>10.4f}")
print("\n三者精度通常很接近(都是优秀的 GBDT 实现); 速度因数据规模/特征数而异")
print("LightGBM 在大数据/高维上常最快(直方图+GOSS+EFB); XGBoost 稳健; 差异更多在易用性/速度")


<a id="4"></a>
## 4. CatBoost 原生类别特征 ⭐ / CatBoost's Native Categoricals

CatBoost 的招牌：**直接吃类别特征**（传 `cat_features`），内部用**有序目标统计(ordered target statistics)** 编码——只用"排在前面的样本"算类别均值，从根上**防止目标泄漏**（5.11 详讲）。对比"丢掉类别 / one-hot+XGBoost / CatBoost 原生"，看类别处理的差别。
CatBoost's signature: **eat categorical features directly** (pass `cat_features`), encoding them internally with **ordered target statistics** — using only "earlier" samples to compute category means, **preventing target leakage** at the root (detailed in 5.11). We compare "drop categorical / one-hot+XGBoost / CatBoost native".


In [ ]:
from sklearn.linear_model import LogisticRegression

# (a) 丢掉类别特征 / drop the categorical
num = ["age","edu_years","hours","capital_gain"]
xgb_drop = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, eval_metric="auc", random_state=0).fit(X_tr[num], y_tr)
auc_drop = roc_auc_score(y_te, xgb_drop.predict_proba(X_te[num])[:,1])

# (b) one-hot + XGBoost (上一节已有) / one-hot + XGBoost
auc_oh = roc_auc_score(y_te, models["XGBoost"].predict_proba(Xte_oh)[:,1])

# (c) CatBoost 原生类别(传 cat_features) / CatBoost native categorical
cat = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=5, cat_features=["occupation"], verbose=0, random_seed=0)
cat.fit(X_tr, y_tr)
auc_cat = roc_auc_score(y_te, cat.predict_proba(X_te)[:,1])

print(f"(a) 丢掉 occupation       test AUC = {auc_drop:.4f}")
print(f"(b) one-hot + XGBoost     test AUC = {auc_oh:.4f}")
print(f"(c) CatBoost 原生类别      test AUC = {auc_cat:.4f}")
print("\n丢掉类别明显更差(信号没了); CatBoost 原生处理省去手工编码, 类别多时尤其方便+常更优")
print("(有序目标统计防泄漏: 编码某行时只用它之前的样本算类别均值, 见 5.11)")


<a id="5"></a>
## 5. 选型指南 + 小结 ⭐ / Selection Guide & Summary

实战怎么选（最实用的产出）：
How to choose in practice (the most useful takeaway):
- **先 LightGBM 做基线**：通常最快，调好 `num_leaves` 就很强，适合快速迭代和大数据。
  **Start with LightGBM:** usually fastest, strong once `num_leaves` is tuned, great for fast iteration and big data.
- **类别特征多/高基数 → CatBoost**：原生处理 + 默认参数就强，省去手工编码。
  **Many/high-cardinality categoricals → CatBoost:** native handling, strong defaults, no manual encoding.
- **要稳健通用/生态成熟 → XGBoost**：长期霸主，文档/部署支持最好。
  **Want robust/general with a mature ecosystem → XGBoost:** the long-time champion, best docs/deployment.
- **追极致精度 → 三个都调一遍再集成**（voting/stacking，8.6/8.7）。
  **For peak accuracy → tune all three and ensemble them** (voting/stacking, 8.6/8.7).

```
三者都是 GBDT(8.4), 区别在工程/算法:
  树生长: XGB level-wise / LGBM leaf-wise(更准易过拟合) / CatBoost 对称树(强正则)
  加速: LGBM 直方图+GOSS+EFB(常最快); XGB 二阶+hist; CatBoost 对称树+GPU
  类别: XGB 需编码; LGBM 原生(直方图); CatBoost 原生(有序目标统计, 防泄漏)
选型: 快速/大数据→LightGBM; 类别多→CatBoost; 通用稳健→XGBoost; 极致→三个集成
精度通常接近; 选型更多看速度/类别/易用性
```

### 💡 面试速查 / Interview cheat-sheet
1. **XGB 稳 / LGBM 快 / CatBoost 擅类别**；都是 GBDT 的优化实现。
   XGB solid / LGBM fast / CatBoost great with categoricals; all optimized GBDT.
2. **leaf-wise(LGBM, 更准易过拟合) vs level-wise(XGB, 平衡) vs 对称树(CatBoost)**。
   leaf-wise (LGBM) vs level-wise (XGB) vs symmetric trees (CatBoost).
3. **LightGBM 快**靠直方图 + GOSS(梯度采样) + EFB(特征捆绑)。
   LightGBM is fast via histograms + GOSS + EFB.
4. **CatBoost 原生类别**用有序目标统计(防泄漏)。
   CatBoost handles categoricals natively via ordered target statistics (leak-free).
5. **三者精度接近**, 选型看速度/类别/易用; 极致用集成(8.6/8.7)。
   Accuracies are close; choose by speed/categoricals/ease; ensemble for peak.

### 下一节 / Next
**8.6 Voting & Averaging**——前面是同质集成(一堆树)。从这课起做**异质集成**: 把逻辑回归/SVM/树等不同模型组合, 硬投票/软投票。
**8.6 Voting & Averaging** — so far homogeneous ensembles (many trees). Now **heterogeneous ensembles**: combine different models (logistic/SVM/trees) via hard/soft voting.
